# Comparación reproducible de modelos de clasificación de noticias

Este notebook descarga la revisión fijada de `mteb/SpanishNewsClassification`, produce las cifras para el resumen del problema, compara tres familias de modelos y exporta toda la evidencia de MLflow.

En Colab: **Entorno de ejecución > Ejecutar todo**. Al final se descargará un ZIP con modelos, métricas, matrices de confusión, reportes y la base local de MLflow.

In [ ]:
!pip -q install mlflow==3.15.2 huggingface_hub pyarrow scikit-learn pandas matplotlib joblib

In [ ]:
from pathlib import Path
import json, shutil, zipfile
import joblib, matplotlib.pyplot as plt, mlflow, pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score, log_loss, top_k_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
DATA_REVISION = '9f568e396857395bd803c63452b83316124d31c9'
EXPERIMENT_NAME = 'spanish-news-model-families'
WORKDIR = Path('/content/spanish_news_model_comparison')
DATASET_PATH = WORKDIR / 'data' / 'train-00000-of-00001.parquet'
OUTPUT_DIR = WORKDIR / 'output'
ARTIFACT_DIR = WORKDIR / 'artifacts'
MLRUNS_DIR = WORKDIR / 'mlruns'
TRACKING_DB = WORKDIR / 'mlflow.db'
LABEL_NAMES = {0:'Alimentación',1:'Astronomía',2:'Economía',3:'Moda',4:'Medicina',5:'Defensa',6:'Motor',7:'Entretenimiento',8:'Política',9:'Religión',10:'Deportes',11:'Tecnología'}
TFIDF = dict(lowercase=True, strip_accents='unicode', ngram_range=(1,2), min_df=2, max_df=0.95, max_features=50000, sublinear_tf=True)
for folder in (WORKDIR, OUTPUT_DIR, ARTIFACT_DIR, MLRUNS_DIR): folder.mkdir(parents=True, exist_ok=True)

In [ ]:
downloaded = hf_hub_download(repo_id='mteb/SpanishNewsClassification', repo_type='dataset', filename='data/train-00000-of-00001.parquet', revision=DATA_REVISION)
shutil.copy2(downloaded, DATASET_PATH)
data = pd.read_parquet(DATASET_PATH)
assert set(data.columns) == {'text','label'}
assert not data[['text','label']].isna().any().any()
assert not data['text'].astype(str).str.strip().eq('').any()
assert data.duplicated(subset=['text']).sum() == 0
assert set(data['label'].unique()) == set(LABEL_NAMES)
lengths = data['text'].str.len()
summary = {'records':len(data),'categories':data['label'].nunique(),'records_per_category_min':int(data['label'].value_counts().min()),'records_per_category_max':int(data['label'].value_counts().max()),'text_length_characters':{'mean':round(float(lengths.mean()),2),'median':int(lengths.median()),'min':int(lengths.min()),'max':int(lengths.max())},'holdout_split':{'train_records':1638,'test_records':410,'strategy':'stratified','random_state':RANDOM_STATE}}
(OUTPUT_DIR/'dataset_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
display(pd.DataFrame([summary['text_length_characters']]))
print(summary)

In [ ]:
def classifier(family):
    if family == 'logistic_regression': return LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_STATE)
    if family == 'complement_naive_bayes': return ComplementNB(alpha=0.3)
    if family == 'linear_svm': return CalibratedClassifierCV(LinearSVC(C=1.0, random_state=RANDOM_STATE), cv=3)
    raise ValueError(family)
def pipeline(family): return Pipeline([('tfidf', TfidfVectorizer(**TFIDF)), ('classifier', classifier(family))])
def top_confusions(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=sorted(LABEL_NAMES)); rows=[]
    for i, actual in enumerate(sorted(LABEL_NAMES)):
        for j, predicted in enumerate(sorted(LABEL_NAMES)):
            if i != j and matrix[i,j]: rows.append({'categoria_real':LABEL_NAMES[actual],'categoria_predicha':LABEL_NAMES[predicted],'casos':int(matrix[i,j])})
    return matrix, pd.DataFrame(rows).sort_values('casos', ascending=False).head(10)
train, test = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data['label'])
mlflow.set_tracking_uri(f'sqlite:///{TRACKING_DB.as_posix()}')
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None: mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=MLRUNS_DIR.as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)
results=[]

In [ ]:
for family in ('logistic_regression','complement_naive_bayes','linear_svm'):
    model = pipeline(family)
    splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv = cross_val_score(model, train['text'], train['label'], cv=splitter, scoring='f1_macro')
    with mlflow.start_run(run_name=family):
        mlflow.log_params({'model_family':family,'tfidf_ngram_range':'(1, 2)','tfidf_max_features':50000,'cv_folds':CV_FOLDS,'random_state':RANDOM_STATE,'holdout_fraction':TEST_SIZE})
        model.fit(train['text'], train['label']); predicted=model.predict(test['text']); probabilities=model.predict_proba(test['text'])
        metrics={'cv_macro_f1_mean':float(cv.mean()),'cv_macro_f1_std':float(cv.std()),'holdout_accuracy':accuracy_score(test['label'],predicted),'holdout_macro_f1':f1_score(test['label'],predicted,average='macro'),'holdout_log_loss':log_loss(test['label'],probabilities,labels=sorted(LABEL_NAMES)),'holdout_top_3_accuracy':top_k_accuracy_score(test['label'],probabilities,k=3,labels=sorted(LABEL_NAMES))}
        mlflow.log_metrics(metrics)
        for step, score in enumerate(cv, 1): mlflow.log_metric('cv_fold_macro_f1', float(score), step=step)
        run_dir=OUTPUT_DIR/family; run_dir.mkdir(exist_ok=True); matrix, confusions=top_confusions(test['label'],predicted)
        (run_dir/'metrics.json').write_text(json.dumps({**metrics,'cv_fold_scores':[float(x) for x in cv]},indent=2),encoding='utf-8'); confusions.to_csv(run_dir/'top_confusions.csv',index=False)
        fig,ax=plt.subplots(figsize=(12,10)); ConfusionMatrixDisplay(matrix,display_labels=[LABEL_NAMES[x] for x in sorted(LABEL_NAMES)]).plot(cmap='Blues',xticks_rotation=45,values_format='d',ax=ax); fig.tight_layout(); fig.savefig(run_dir/'confusion_matrix.png',dpi=160); plt.close(fig)
        joblib.dump(model,run_dir/'model.joblib')
        for artifact in run_dir.iterdir(): mlflow.log_artifact(str(artifact))
    results.append({'family':family,**metrics}); print(family, metrics)

In [ ]:
comparison=pd.DataFrame(results).sort_values('cv_macro_f1_mean',ascending=False)
comparison.to_csv(OUTPUT_DIR/'comparison_summary.csv',index=False)
display(comparison.round(4))
print('Modelo seleccionado por F1 macro CV:', comparison.iloc[0]['family'])
archive=Path('/content/evidencia_comparacion_modelos_mlflow.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zip_file:
    for path in [OUTPUT_DIR, MLRUNS_DIR]:
        for file in path.rglob('*'):
            if file.is_file(): zip_file.write(file,file.relative_to(WORKDIR))
    zip_file.write(TRACKING_DB,TRACKING_DB.relative_to(WORKDIR))
from google.colab import files
files.download(str(archive))